# **Smartphone Behaviour and Student Stress: A CRISP-DM Analysis of StudentLife**

## DBMS71008 Data Analytics | CA2 CRISP-DM Analytics Project | Daniel Courtney

## **Phase 1: Business Understanding**

### **1.1 Context and Problem**

University students face recurring periods of high academic pressure, and stress tends to rise as workload increases across the academic term. Heavy smartphone use and poor sleep are commonly suggested as contributing factors. However, student support services typically learn about difficulties only when students self report them, after problems have escalated, with much of this stress remaining hidden from staff.

Smartphones offer a way to observe daily behaviour. This project investigates whether behavioural signals captured by smartphones are associated with students self-reported stress, using the *StudentLife dataset*. *(Wang et al., 2014)*


### **1.2 Stakeholders and Users**

| Stakeholder | Role | Interest / How they are affected |
| --- | --- | --- |
| StudentLife Participants | Supply data | Behaviour and stress data analysed for improved supports |
| Student Support Services | Primary User | Want insights into when students may need support, to guide outreach and resources |
| StudentLife Research Team | Data Owner | Set terms of use and attribution for the dataset |
| Researchers | Secondary Users | Interested in whether sensing can support wellbeing research |
| Data Protection Officer | Governance | Decide whether monitoring is acceptable, and under what consent, privacy, and data protection rules. |
| Current Students | Affected Party | Could benefit from earlier, better-timed support. But risks being wrongly flagged (false positive), missed when struggling (false negative), or feeling monitored |

### **1.3 Business or Practical Objective**

**Practical Objective:**

Help student support services understand which daily behaviours are associated with elevated stress, so they can decide when and how to offer proactive support.

### **1.4 Analytics Objective**

**Question**

To what extent can passively sensed smartphone usage, sleep, and physical activity distinguish days on which StudentLife participants report elevated stress from days on which they do not?


**Analytics Objective**

Build a binary classifier that distinguishes high stress from other student-days using daily phone use, sleep, and activity features (inputs model will use to make a decision), and evaluate whether it outperforms a majority-class baseline (predict whichever class is most common) in recall and F1 for high-stress days on students not seen during training.

Recall is essential as it means support services missing a struggling student (false negative) is wose than an unnecessary check in (false positive) *(Adler et al., 2022)*. 


**Unit of Analysis**

One student on one calendar day.

### **1.5 Success Criteria**

**Technical Criteria**

| Criterion | How it will be measured |
| --- | --- |
| Beats the baseline | Recall and F1 for high-stress days exceed a majority-class baseline |
| Generalises to new students | Evaluated on students held out from training |

<br>

**Practical Criteria**

| Criterion | How it will be measured |
| --- | --- |
| Interpretable | Main patterns can be explained in plain language to non-technical support staff |
| Useful for the intended decision | Findings indicate which behaviours or periods relate to elevated stress, informing outreach timing |
| Appropriate use | Results describe patterns, not labels on individual students |
| Honest limitations | Sample size, single institution and 2013 data clearly stated |

<br>

Recall is prioritised over accuracy because, for support services, missing a genuinely stressed student *(a false negative)* is far more costly than an unnecessary check-in *(a false positive)*. F1 is reported alongside it so that precision is not ignored.

### **1.6 Constraints and Assumptions**

**Constraints**

| Type | Constraint |
| --- | --- |
| Time | Individual project, with final submission due 4 December 2026 |
| Data | Limited to the existing StudentLife data (48 students, one class, Spring 2013), no new data can be collected |
| Scope | Stress as the only outcome. Phone use, sleep and activity as the only feature areas. Location data excluded |

<br>

**Assumptions**

| Assumption | Why it matters |
| --- | --- |
| Self-reported stress reflects how stressed students actually felt | It is the target label. If it is inaccurate, the model learns noise |
| Sensor inferences (activity, sleep) are accurate enough for daily summaries | They are classifier outputs with known error |
| Students carried their study phone throughout the day | Days where they did not would understate behaviour |
| Student-days with sufficient data are representative of the term | If missing data is concentrated in stressful weeks, high-stress days are under-represented |

<br>

**Risks**

| Risk | Why it matters | Mitigation |
| --- | --- | --- |
| Sparse stress responses | Students did not answer every day, meaning fewer labelled rows | Drop or flag students with too few responses |
| Missing sensor data | Phone off, uncharged, or not carried | Exclude days with too little data |
| Ethics and Privacy | Behavioural and mental health data about real people | Frame the findings as patterns, not individual labels |

### **1.7 Initial Project Plan**

**Proposed Technique:** 

Binary classification using a decision tree, compared against a majority class baseline and an alternative model, evaluated with recall and F1 on a student-level split.

<br>

**Tools:** 

Python, Jupyter Notebook, Github

<br>

**CRISP-DM Activities**

| Phase | Activities | Target Dates |
| --- | --- | --- |
| *Business Understanding* | Problem, stakeholders, objectives, success criteria, constraints | 09 Nov |
| *Data Understanding* | Feasibility check (stress responses per student, sensor coverage), data description, exploration and quality report, data dictionary | 10-16 Nov |
| *Data Preparation* | Recode stress scale, aggregate sensor data to student-days, join with stress labels, handle missing data, preparation log, before/after validation | 10-16 Nov |
| *Modelling* | Student-level train/test split, baseline, decision tree | 16-23 Nov |
| *Evaluation* | Metrics against success criteria, error analysis, limitations, bias and ethics | 23-30 Nov |
| *Deployment & submission* | Recommendations and monitoring, reflection, AI collaboration statement, final paper | 30 Nov -04 Dec |

**Checkpoint:** if the prepared student-day dataset is not complete by 20 November, scope will be reduced (fewer variables or students) rather than extending preparation, following lecturer feedback (10 Nov 2026).

*The plan is to be revised as the project develops, with changes and their reasoning recorded accordingly*

<br>

**Project Feedback**

| Date | Source | Feedback | How it was addressed |
| --- | --- | --- | --- |
| 10 Nov 2026 | Email, Andrew Shields | Continue with StudentLife,  combining JSON/CSV files into a student-day table is valid Data Preparation. Keep scope manageable, document every transformation step, and prioritise answering the question over data wrangling. Reduce variables or students rather than switching datasets if needed. | Kept StudentLife. Limited to four data streams, transformation steps recorded in the preparation log. |

## **Phase 2: Data Understanding**

### **Environment Setup**

In [7]:
# data wrangling
import pandas as pd
import numpy as np
from pathlib import Path

# data viz
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


### **2.1 Initial Data Collection Report**

**2.1.1 Source and Permissions**

| Type | Details |
| --- | --- |
| Data Source | Kaggle dataset StudentLife https://www.kaggle.com/datasets/dartweichen/student-life?select=dataset |
| Owner / Publisher | StudentLife Dartmouth College (Wang et al., 2014) – https://StudentLife.cs.dartmouth.edu/ |
| Access Date | 29 October 2026 |
| Licence / Permission | No explicit licence is stated on the Kaggle page or the original Dartmouth dataset page. The Dartmouth team publicly releases the anonymised dataset for research, and asks that Wang et al. (2014) is cited if the dataset is used in a publication (StudentLife, n.d.). Use in this project is non-commercial and academic, the dataset is cited throughout, and raw data is not redistributed in the repository. |

<br>

**2.1.2 Collection Process**

| Type | Details |
| --- | --- |
| Collection Process | A StudentLife android app passively collected sensor based data such as activity, sleep, and location, and delivered EMA surveys. Pre and post term surveys were conducted via SurveyMonkey. |

<br>

**2.1.3 Population and Scope**

| Type | Details |
| --- | --- |
| Population | 48 Dartmouth students who completed the study |
| Time Period | Spring academic term 2013, for a period of 10 weeks. Approx late March to early June  |
| Geography | Dartmouth College, New Hampshire, USA. |
| Scope | **Dataset:** passive sensing inferences (activity, conversation, phone lock/charge, light, location), EMA responses (stress, mood, sleep, social and others, EMA and mood responses), pre/post mental health surveys, and education data (grades, deadlines) – 52.6 GB of sensing data in total (Wang et al., 2014). **This project:** stress EMA (target), phone lock, activity and one sleep measure, with all other data excluded. |

<br>

**2.1.4 Exclusions and Limitations**

| Type | Details |
| --- | --- |
| Known Exclusions | Students who dropped out of the study or the class were removed (60 joined, 48 completed). |
| Aggregation | The sensor data are inferences from phone classifiers, not raw sensor readings. |
| Capping | None identified. |
| Sampling | Voluntary sample from one class, not random. |
| Access Limitations | Kaggle repost of the original |

<br>

**2.1.5 Files Acquired**

*AI-assisted:* The selection of files below was developed with the help of Claude (Anthropic, Opus 5.5). I provided my research question and the dataset link, and the tool identified which folders correspond to each part of the question, described their file formats and key fields, and suggested how they link together. I verified this by opening each file myself, confirming the field names and the stress answer coding against `EMA/EMA_definition.json`, and checking the activity codes against Wang et al. (2014). The final choice of files, including the sleep measure, is my own decision.

| Required from Question | File / folder | Contents | Key fields | How it relates |
| --- | --- | --- | --- | --- |
| Elevated stress (target) | `EMA/response/Stress/` | 49 JSON files, one per student (`Stress_uXX.json`), containing stress EMA responses | `level` (1–5, stored as text), `resp_time` (Unix seconds, UTC), `location`, student ID from filename | Target label, recoded to high / not-high stress and joined to features on student ID + date |
| Stress answer coding | `EMA/EMA_definition.json` | Wording and answer options for every EMA question | `name`, `question_id`, `question_text`, `options` | Confirms the stress scale is not ordered by severity (1 A little stressed … 5 Feeling great), so it must be recoded |
| Smartphone usage | `sensing/phonelock/` | 49 CSV files, one per student (`phonelock_uXX.csv`), listing periods the phone was locked | `start`, `end` (Unix seconds, UTC) | Aggregated to daily lock/unlock time per student; joined on student ID + date |
| Physical activity | `sensing/activity/` | 49 CSV files, one per student (`activity_uXX.csv`), with activity inferences every few seconds | `timestamp` (Unix seconds), `activity inference` (0 stationary, 1 walking, 2 running, 3 unknown; Wang et al., 2014) | Aggregated to daily active minutes; joined on student ID + date |
| Sleep (option A: self-report) | `EMA/response/Sleep/` | 49 JSON files, one per student, with sleep EMA responses | `hour` (codes 1–19 = <3 h to 12 h), `rate`, `social`, `resp_time` | Daily sleep duration; joined on student ID + date (covers 867 of the 1,264 stress student-days) |
| Sleep (option B: sensor proxy) | `sensing/dark/`, `sensing/phonecharge/` (with `phonelock`) | 49 CSV files each, listing periods the phone was in darkness / charging | `start`, `end` (Unix seconds, UTC) | Combined to estimate an overnight sleep window per student; joined on student ID + date |

All files are linked by student ID (`u00`–`u59`, from the filename) and the local
date derived from the Unix timestamps.

### **2.2 Data Description Report**

This section describes the raw data as acquired, before any cleaning. Issues noticed here are recorded and carried forward to the *Data Quality Report* and addressed in *Data Preparation*.

In [ ]:
from pathlib import Path
import kagglehub

# Using kagglehub to so that the dataset will download when this code is run.
path = Path(kagglehub.dataset_download("dartweichen/student-life",
                                       output_dir="data/raw/student-life"))
DATA = next(path.rglob("EMA_definition.json")).parent.parent
print(DATA, DATA.exists())

**2.2.1 Dataset Dimensions and Unit of Analysis**

<br>

**2.2.2 Variable Names, Meanings, Data Types, Units, and Valid Ranges**

<br>

**2.2.3 Identifiers, Candidate Inputs, and Potential Target or Outcome**

<br>

**2.2.4 Data Dictionary or Equivalent Table**

### **2.3 Data Exploration Report**

### **2.4 Data Quality Report**